# How to Train YOLO26 Object Detection on a Custom Dataset

---

[![roboflow](https://raw.githubusercontent.com/roboflow-ai/notebooks/main/assets/badges/roboflow-blogpost.svg)](https://blog.roboflow.com/how-to-train-yolo26-custom-data/) [![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/ultralytics/ultralytics)

YOLO26 introduces a unified architecture designed to support detection, segmentation, and pose tasks within a single model family. The model uses an anchor-free design with a decoupled head.


**NOTE:** To make it easier for us to manage datasets, images and models we create a `HOME` constant.

In [ ]:
from pathlib import Path

HOME = Path.cwd().resolve()
while HOME.name == 'datasets':
    HOME = HOME.parent
print(HOME)

### Install dependencies required for YOLO26


In [ ]:
%pip install -q --index-url https://download.pytorch.org/whl/cu124 --extra-index-url https://pypi.org/simple torch==2.6.0+cu124 torchvision==0.21.0+cu124 torchaudio==2.6.0+cu124
%pip install -q "ultralytics>=8.4.0" supervision roboflow python-dotenv pyyaml scipy requests_toolbelt tqdm typer defusedxml

# prevent ultralytics from tracking your activity
!yolo settings sync=False
import ultralytics
ultralytics.checks()

### Download example data

Downloads example images for testing. You can use these or replace them with your own images.

In [ ]:
!powershell -Command "Invoke-WebRequest -Uri 'https://media.roboflow.com/notebooks/examples/dog-2.jpeg' -OutFile 'dog-2.jpeg'"
!powershell -Command "Invoke-WebRequest -Uri 'https://media.roboflow.com/notebooks/examples/dog-3.jpeg' -OutFile 'dog-3.jpeg'"

In [ ]:
file = "dog-2.jpeg"
file_source = HOME / file

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26m.pt")
if model is not None:
    print("Model loaded successfully.")

In [ ]:
results = model.predict(
    source=file_source,
    save=True,
    verbose=False,
    project=HOME / "runs" / "detect",
    name="predict",
    exist_ok=True,
)
prediction = results[0]
prediction_dir = Path(prediction.save_dir)
print(f"Predictions saved to {prediction_dir}")

**NOTE:** Result annotated image got saved in `{HOME}/runs/detect/predict/`. Let's display it.

In [ ]:
prediction_dir = Path(prediction_dir)

if not prediction_dir.exists():
    print(f"Prediction directory not found: {prediction_dir}")
else:
    print(f"Listing {prediction_dir}")
    for item in sorted(prediction_dir.iterdir()):
        print(item.name)

In [ ]:
from IPython.display import Image as IPyImage

filename = str(prediction_dir / f"{Path(file).stem}.jpg")
IPyImage(filename=filename, width=600)

### SDK

In [ ]:
from ultralytics import YOLO
from PIL import Image

model = YOLO('yolo26m.pt')
image = Image.open(f'{HOME}/{file}')
result = model.predict(image, verbose=False)[0]

**NOTE:** The obtained `result` object stores information about the location, classes, and confidence levels of the detected objects.

In [ ]:
result.boxes.xyxy

In [ ]:
result.boxes.conf

In [ ]:
result.boxes.cls

**NOTE:** YOLO26 can be easily integrated with `supervision` using the familiar `from_ultralytics` connector.

In [ ]:
import supervision as sv

detections = sv.Detections.from_ultralytics(result)

In [ ]:
import supervision as sv
from PIL import Image

def annotate(image: Image.Image, detections: sv.Detections) -> Image.Image:
    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)

    box_annotator = sv.BoxAnnotator()
    label_annotator = sv.LabelAnnotator(
        text_color=sv.Color.BLACK,
        text_scale=text_scale,
        smart_position=True
    )

    out = image.copy()
    out = box_annotator.annotate(out, detections)
    out = label_annotator.annotate(out, detections)
    out.thumbnail((1000, 1000))
    return out

In [ ]:
annotated_image = annotate(image, detections)
annotated_image

## Fine-tune YOLO26 on custom dataset

**NOTE:** When training YOLO26, make sure your data is located in `datasets`. If you'd like to change the default location of the data you want to use for fine-tuning, you can do so through Ultralytics' `settings.json`. In this tutorial, we will use one of the [datasets](https://universe.roboflow.com/liangdianzhong/-qvdww) available on [Roboflow Universe](https://universe.roboflow.com/). When downloading, make sure to select the `yolov11` export format.

In [ ]:
import os

datasets_dir = HOME / "datasets"
datasets_dir.mkdir(parents=True, exist_ok=True)
print(datasets_dir)

In [ ]:
import os
import importlib
import urllib3
from dotenv import load_dotenv
from roboflow import Roboflow

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
# Roboflow's SDK uses requests without a certificate override in this environment.
# This keeps the tutorial working on Windows networks that intercept HTTPS traffic.
import requests
import requests.sessions as requests_sessions
requests_sessions = importlib.reload(requests_sessions)
_requests_request = requests_sessions.Session.request

def _request_without_ssl_verification(self, method, url, **kwargs):
    kwargs.setdefault('verify', False)
    kwargs.setdefault('timeout', 30)
    return _requests_request(self, method, url, **kwargs)

requests_sessions.Session.request = _request_without_ssl_verification
requests.Session.request = _request_without_ssl_verification

load_dotenv(HOME / '.env')
ROBOFLOW_API_KEY = os.getenv('ROBOFLOW_API_KEY')
if not ROBOFLOW_API_KEY:
    raise ValueError("ROBOFLOW_API_KEY is not set in notebooks/.env")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

In [ ]:
dataset_dir = datasets_dir / 'basketball-player-detection-3-14'
dataset_dir.mkdir(parents=True, exist_ok=True)
workspace = rf.workspace("roboflow-jvuqo")
project = workspace.project("basketball-player-detection-3-ycjdo")
version = project.version(14)
dataset = version.download("yolo26", location=str(dataset_dir), overwrite=True)
print(dataset.location)

## Custom Training

In [ ]:
import torch
from pathlib import Path
from ultralytics import YOLO

data_root = Path(dataset.location)
data_yaml = data_root / 'data.yaml'
if not data_yaml.exists():
    raise FileNotFoundError(f"Dataset config not found: {data_yaml}. Re-run the download cell and confirm the dataset extracted correctly in {data_root}.")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available in the current Python environment. Install a CUDA-enabled PyTorch build and reopen the notebook kernel before training.")

# High-capacity training profile for RTX 4090 + 128GB RAM + full-HD images.
# Start with a larger model and higher resolution, then reduce imgsz or batch if VRAM becomes tight.
model = YOLO('yolo26x.pt')
model.train(
    data=str(data_yaml),
    epochs=150,
    imgsz=1280,
    batch=4,
    device=0,
    workers=16,
    cache='ram',
    pretrained=True,
    optimizer='AdamW',
    lr0=0.005,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=50,
    mosaic=1.0,
    mixup=0.1,
    degrees=5.0,
    scale=0.5,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    hsv_v=0.25,
    label_smoothing=0.05,
    close_mosaic=10,
    plots=True,
    exist_ok=True,
)